# 🧬 GIADA Task 3 — Cella congiunta `m+h`
Confronto preregistrato fra trunk indipendenti e rappresentazione di voltaggio condivisa, sotto voltage clamp costante.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys
WORKSPACE=Path('/kaggle/working/giada_task_3'); GIADA_REPO=WORKSPACE/'giada'; TEACHER_REPO=WORKSPACE/'neuron_as_deep_net'
OUTPUT_DIR=Path('/kaggle/working/artifacts/giada_joint_m_h_cell')
def run(args,cwd=None): print('+',' '.join(map(str,args))); subprocess.run(list(map(str,args)),cwd=cwd,check=True)


In [ ]:
WORKSPACE.mkdir(parents=True,exist_ok=True)
if not GIADA_REPO.exists(): run(['git','clone','--branch','codex/surrogate-validity-audit','--single-branch','https://github.com/Zagred47/giada.git',GIADA_REPO])
else: run(['git','fetch','origin','codex/surrogate-validity-audit'],cwd=GIADA_REPO); run(['git','checkout','--detach','FETCH_HEAD'],cwd=GIADA_REPO)
if not TEACHER_REPO.exists(): run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',TEACHER_REPO])
run(['git','checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],cwd=TEACHER_REPO)
REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=GIADA_REPO,text=True).strip(); print('GIADA revision:',REVISION)


In [ ]:
sys.path.insert(0,str(GIADA_REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]: del sys.modules[name]
import torch
assert torch.cuda.is_available(),'La Task 3 registrata richiede una GPU CUDA Kaggle.'
from src.giada_teacher import ExtractedGateFormula,JointGateCellConfig,prepare_joint_gate_dataset,run_joint_gate_playground,evaluate_joint_gate_playground
task1b=json.loads((GIADA_REPO/'experiments/teacher_gate_m_diagnosis_result_v1.json').read_text())
task2b=json.loads((GIADA_REPO/'experiments/teacher_gate_h_identifiability_result_v1.json').read_text())
prereg=json.loads((GIADA_REPO/'experiments/teacher_joint_gate_cell_preregistration_v1.json').read_text())
assert task1b['decision']['engineering_continuation_gate_2_5e_3_passed'] and task2b['task3_authorized']
display({'task1b_m_debt':task1b['selected_candidate']['fresh_rmse_by_stratum'],'task2b_h':task2b,'preregistration':prereg})


In [ ]:
assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
formula=ExtractedGateFormula.from_mod(TEACHER_REPO/'L5PC_NEURON_simulation/mods/Ca_HVA.mod')
bundle=prepare_joint_gate_dataset(formula); config=JointGateCellConfig()
display({'contract':bundle['contract'],'families':config.families,'checkpoints':config.checkpoints})


In [ ]:
training_report=run_joint_gate_playground(bundle,OUTPUT_DIR,config)
display({'valid':training_report['valid'],'winner':training_report['winner'],'selection':training_report['selection']})


In [ ]:
final_report=evaluate_joint_gate_playground(bundle,OUTPUT_DIR,config)
display({'valid':final_report['valid'],'decision':final_report['decision'],'fresh_not_used_for_selection':not final_report['selection_used_fresh']})
assert final_report['valid'] and not final_report['selection_used_fresh']


## 📦 Download robusto
Metodo Blob/base64 previsto dal progetto.

In [ ]:
from base64 import b64encode
from IPython.display import Javascript,display
archive=Path(shutil.make_archive('/kaggle/working/giada_joint_m_h_cell','zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name))
payload=b64encode(archive.read_bytes()).decode('ascii'); filename=archive.name
display(Javascript(f"""const raw=atob('{payload}');const bytes=new Uint8Array(raw.length);for(let i=0;i<raw.length;i++)bytes[i]=raw.charCodeAt(i);const url=URL.createObjectURL(new Blob([bytes],{{type:'application/zip'}}));const a=document.createElement('a');a.href=url;a.download='{filename}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),10000);"""))
print('Download avviato:',filename,archive.stat().st_size,'byte')
